In [6]:
# %%
import pickle
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# ⏱️ 量化时光机：机构级每日战报
# ==========================================
TARGET_DATE = "2026-04-22" 
report_file = f"Daily_Reports/Report_{TARGET_DATE}.pkl"

try:
    with open(report_file, 'rb') as f:
        snap = pickle.load(f)
        
    # 标题头
    display(Markdown(f"# 📊 宽客实战日报 | 交易日: {snap['date']}"))
    display(Markdown(f"### 👑 今日市场绝对主线 (King Factor): **`{snap['king_factor']}`**"))
    display(Markdown("---"))
    

    # 模块 1：OLS 市场风向归因
    r2_fac = snap['ols_report']['r2_factors'] * 100
    r2_ind = snap['ols_report']['r2_ind'] * 100
    display(Markdown(f"#### 📡 1. 市场微观结构探测 (解释力 - 因子: `{r2_fac:.2f}%` | 行业: `{r2_ind:.2f}%`)"))
    
    # ✨ 修复了 Pandas 的版本兼容警告：用 map 替代了 applymap
    style_ols = snap['ols_report']['stats_df'].style.map(
        lambda x: 'color: red; font-weight: bold' if isinstance(x, str) and '追捧 ★' in x else 
                  ('color: green; font-weight: bold' if isinstance(x, str) and '抛售 ★' in x else ''),
        subset=['风向状态']
    ).background_gradient(subset=['Beta系数'], cmap='coolwarm', vmin=-0.05, vmax=0.05)
    display(style_ols)
    
    # 模块 1.5：因子强度“英雄榜”
    display(Markdown(f"#### 🏆 1.5 因子全维度强度对比 (当前选股基准: `{snap['king_factor']}`)"))
    
    # 对统计表进行美化：Beta 用渐变色，T值绝对值大于 1.96 的加粗
    king_stats_styled = snap['king_stats'].style.background_gradient(
        subset=['Beta (强度)'], cmap='RdYlGn'
    ).map(
        lambda x: 'font-weight: bold; color: #FF4500' if isinstance(x, (int, float)) and abs(x) > 1.96 else '',
        subset=['T值 (显著性)']
    )
    
    display(king_stats_styled)
    
    # 计算次强因子与强度差距
    stats_df = snap['king_stats']
    if len(stats_df) > 1:
        top_1 = stats_df.iloc[0]
        top_2 = stats_df.iloc[1]
        gap = round(top_1['Beta (强度)'] - top_2['Beta (强度)'], 4)
        display(Markdown(f"> 💡 **复盘笔记**：今日最强因子 `{top_1['因子名称']}` 与次强因子 `{top_2['因子名称']}` 的强度差距为 **`{gap}`**。"))


    # 模块 2：强势板块与龙头画像
    display(Markdown("#### 🎯 2. 强势板块下钻与龙头画像"))
    # 横向展示领涨和领跌板块
    col1, col2 = snap['drill_report']['top_industries'], snap['drill_report']['bottom_industries']
    display(pd.concat([col1.set_index('板块名称'), col2.set_index('板块名称')], axis=1, keys=['🔥 领涨板块', '❄️ 领跌板块']))
    
    display(Markdown("**【强势板块内部资金画像】**"))
    display(snap['drill_report']['portraits_df'].style.set_properties(**{'text-align': 'left'}))
    display(Markdown("---"))


    # 模块 3：明日交易底仓狙击池
    display(Markdown(f"#### 🔫 3. 明日开盘狙击池 (按核心因子 **`{snap['king_factor']}`** 降序排列)"))


    display_cols = ['code', 'name', snap['king_factor'], 'Momentum', 'Liquidity', 'Size']
    # 将更多你想看的因子加到表格里显示 (比如把 Amihud 也展示出来)：
    # display_cols = ['code', 'name', snap['king_factor'], 'Momentum', 'Liquidity', 'Size', 'Amihud']
    
    display_cols = list(dict.fromkeys(display_cols)) 
    
    # 原来是 .head(10)，改成 .head(20) 就能看前 20 名金股！
    top_picks = snap['top_picks'].head(20)[display_cols] 
    
    display(top_picks.style.background_gradient(subset=[snap['king_factor']], cmap='YlOrRd'))



except FileNotFoundError:
    print(f"❌ 找不到 {TARGET_DATE} 的战报！")
# %%

# 📊 宽客实战日报 | 交易日: 2026-04-22

### 👑 今日市场绝对主线 (King Factor): **`Liquidity`**

---

#### 📡 1. 市场微观结构探测 (解释力 - 因子: `21.54%` | 行业: `27.99%`)

,因子 (Factor),Beta系数,T值 (显著性),风向状态
0,Momentum,19.892000,3.890000,追捧 ★
1,Short_Rev,-2.922000,-0.540000,抛售
2,Low_Vol,-30.989000,-5.540000,抛售 ★
3,Liquidity,30.414000,5.880000,追捧 ★
4,Size,3.416000,0.630000,追捧
5,Value_BP,-22.440000,-3.040000,抛售 ★
6,Mom_Sharpe,20.439000,3.840000,追捧 ★
7,Vol_Price_Corr,-2.240000,-0.430000,抛售
8,Amihud,-2.731000,-0.490000,抛售


...

**【强势板块内部资金画像】**

,强势板块,代码,名称,预期超额涨幅,核心因子画像
0,M75科技推广和应用服务业,sz.003035,南网能源,+8.61%,低Low_Vol(-1.0)
1,C39计算机、通信和其他电子设备制造业,sz.002938,鹏鼎控股,+28.21%,"高Momentum(+1.9), 低Short_Rev(-1.9), 高Size(+1.1), 高Mom_Sharpe(+1.8), 高Vol_Price_Corr(+1.3)"
2,C39计算机、通信和其他电子设备制造业,sz.002384,东山精密,+27.81%,"高Momentum(+3.3), 低Short_Rev(-3.3), 低Low_Vol(-2.1), 高Liquidity(+2.5), 高Size(+1.5), 低Value_BP(-1.0), 高Mom_Sharpe(+2.6), 低Amihud(-1.3)"
3,C39计算机、通信和其他电子设备制造业,sz.300502,新易盛,+25.30%,"高Momentum(+3.3), 低Low_Vol(-1.9), 高Liquidity(+1.8), 高Size(+2.5), 低Value_BP(-1.1), 高Mom_Sharpe(+2.4), 高Vol_Price_Corr(+1.5), 低Amihud(-1.4)"
4,C39计算机、通信和其他电子设备制造业,sz.001389,广合科技,+23.98%,"高Momentum(+2.2), 低Low_Vol(-2.3), 高Liquidity(+3.4), 微盘股(-1.1), 低Value_BP(-1.0), 高Mom_Sharpe(+1.5)"
5,C39计算机、通信和其他电子设备制造业,sh.600707,彩虹股份,+23.23%,"高Momentum(+1.8), 低Short_Rev(-3.3), 低Low_Vol(-1.3), 高Mom_Sharpe(+1.5), 高Amihud(+2.3)"
6,C39计算机、通信和其他电子设备制造业,sz.003031,中瓷电子,+21.41%,"高Momentum(+3.3), 低Short_Rev(-2.3), 低Low_Vol(-1.8), 高Liquidity(+1.7), 高Mom_Sharpe(+2.8), 高Vol_Price_Corr(+1.2)"
7,C39计算机、通信和其他电子设备制造业,sh.601231,环旭电子,+20.92%,中庸(随板块普涨)
8,C39计算机、通信和其他电子设备制造业,sh.688220,翱捷科技,+20.71%,低Low_Vol(-1.1)
9,C39计算机、通信和其他电子设备制造业,sh.603920,世运电路,+16.89%,高Liquidity(+1.0)


---

#### 🔫 3. 明日开盘狙击池 (按核心因子 **`Liquidity`** 降序排列)

,code,name,Liquidity,Momentum,Size
0,2460,赣锋锂业,3.259625,1.127309,0.845012
1,300394,天孚通信,2.140348,1.733490,1.987119
2,300274,阳光电源,1.960502,-2.015048,1.768710
3,2466,天齐锂业,1.814478,2.730712,0.755016
4,977,浪潮信息,1.802107,1.920454,0.838514
5,300502,新易盛,1.427986,2.383854,2.583970
6,300567,精测电子,1.289066,0.216839,-0.548610
7,300014,亿纬锂能,1.256248,-0.317936,1.246004
8,688347,华虹公司,1.123055,0.634732,0.039414
9,688008,澜起科技,1.004130,1.122171,1.448619
